<a href="https://colab.research.google.com/github/fhhjhhg/A2A/blob/main/Copy_of_HYPIR_SD2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p align="center">
    <img src="https://github.com/XPixelGroup/HYPIR/blob/main/assets/logo.png?raw=true" width="400">
</p>

## HYPIR: Harnessing Diffusion-Yielded Score Priors for Image Restoration

![visitors](https://visitor-badge.laobi.icu/badge?page_id=XPixelGroup/HYPIR) [![Try a demo on Replicate](https://replicate.com/0x3f3f3f3fun/hypir-sd2/badge)](https://replicate.com/0x3f3f3f3fun/hypir-sd2) [![Open in OpenXLab](https://cdn-static.openxlab.org.cn/app-center/openxlab_app.svg)](TODO)

Xinqi Lin<sup>1,2</sup>, [Fanghua Yu](https://github.com/Fanghua-Yu)<sup>1</sup>, Jinfan Hu<sup>1,2</sup>, [Zhiyuan You](https://zhiyuanyou.github.io/)<sup>1,3</sup>, Wu Shi<sup>1</sup>, [Jimmy S. Ren](https://www.jimmyren.com/)<sup>4,5</sup>, [Jinjin Gu](https://www.jasongt.com/)<sup>6,\*</sup>, [Chao Dong](https://scholar.google.com.hk/citations?user=OSDCB0UAAAAJ)<sup>1,\*</sup>

\*: Corresponding author

In [ ]:
from google.colab import ai

stream = ai.generate_text("Tell me a short story.", stream=True)
for text in stream:
  print(text, end='')

In [ ]:
from google.colab import ai

stream = ai.generate_text("Tell me a short story.", stream=True)
for text in stream:
  print(text, end='')

In [ ]:
from google.colab import ai

stream = ai.generate_text("Tell me a short story.", stream=True)
for text in stream:
  print(text, end='')

In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

worksheet = gc.open('Your spreadsheet name').sheet1

# get_all_values gives a list of rows.
rows = worksheet.get_all_values()
print(rows)

# Convert to a DataFrame and render.
import pandas as pd
pd.DataFrame.from_records(rows)

In [ ]:
# https://pypi.python.org/pypi/libarchive
!apt-get -qq install -y libarchive-dev && pip install -U libarchive
import libarchive

In [ ]:
# Import PyDrive and associated libraries.
# This only needs to be done once per notebook.
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authenticate and create the PyDrive client.
# This only needs to be done once per notebook.
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

# List .txt files in the root.
#
# Search query reference:
# https://developers.google.com/drive/v2/web/search-parameters
listed = drive.ListFile({'q': "title contains '.txt' and 'root' in parents"}).GetList()
for file in listed:
  print('title {}, id {}'.format(file['title'], file['id']))

In [ ]:
# Import PyDrive and associated libraries.
# This only needs to be done once per notebook.
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authenticate and create the PyDrive client.
# This only needs to be done once per notebook.
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

# List .txt files in the root.
#
# Search query reference:
# https://developers.google.com/drive/v2/web/search-parameters
listed = drive.ListFile({'q': "title contains '.txt' and 'root' in parents"}).GetList()
for file in listed:
  print('title {}, id {}'.format(file['title'], file['id']))

In [ ]:
# Import PyDrive and associated libraries.
# This only needs to be done once per notebook.
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authenticate and create the PyDrive client.
# This only needs to be done once per notebook.
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

# List .txt files in the root.
#
# Search query reference:
# https://developers.google.com/drive/v2/web/search-parameters
listed = drive.ListFile({'q': "title contains '.txt' and 'root' in parents"}).GetList()
for file in listed:
  print('title {}, id {}'.format(file['title'], file['id']))

### Mount Google Drive

Run the following cell to mount your Google Drive. This will prompt you to authenticate your Google account.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Once your Drive is mounted, you can access your files from `/content/drive/My Drive/`. For example, to list the contents of your Drive, you can use:

```bash
!ls '/content/drive/My Drive/'
```

Or, if you know the path to your data, you can load it directly. For instance, to load a CSV file named `my_data.csv` located in the root of your 'My Drive' folder:

```python
import pandas as pd
df = pd.read_csv('/content/drive/My Drive/my_data.csv')
print(df.head())
```

Please let me know the path to your data once your Drive is mounted, and I can help you load it.

# 1. Preparations

1. Make sure you choose **GPU** as hardware. Free T4 GPU is good enough for running HYPIR-SD2 model.
2. Clone the repository, install environment and download pre-trained weight.

In [ ]:
!rm -rf HYPIR
!git clone https://github.com/XPixelGroup/HYPIR.git
%cd HYPIR

In [ ]:
!pip install -r requirements.txt
import os
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

# Define the models to download
models = [
    ("https://huggingface.co/lxq007/HYPIR/resolve/main/HYPIR_sd2.pth", "HYPIR_sd2.pth")
]

def download_model(url, fname):
    subprocess.run(["wget", "-q", url, "-O", fname], check=True)
    return fname

# Download models in parallel
with ThreadPoolExecutor(max_workers=1) as executor:
    futures = {executor.submit(download_model, url, fname): fname for url, fname in models}
    for future in as_completed(futures):
        future.result()


# 2. Launch gradio app

1. Run the next code block to launch gradio app. You'll need to **wait a few minutes** during the initial load, as the program downloads the Hugging Face pre-trained model.

    During program execution, some information will pop up:

    ```txt
    Max size set to (2048, 2048), max pixels: 4194304
    ...
    Load model weights from HYPIR_sd2.pth
    ...
    Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
    * Running on public URL: https://58899f87fcc429eb66.gradio.live
    ...
    ```

    You'll see a public URL, **click on it to access the Gradio page**. Hope you enjoy this model.
2. ⭐If HYPIR is helpful for you, please help star this [repo](https://github.com/XPixelGroup/HYPIR). Thanks!🤗

In [ ]:
import random
import os
from argparse import ArgumentParser

import gradio as gr
import torchvision.transforms as transforms
from accelerate.utils import set_seed
from omegaconf import OmegaConf
from PIL import Image

from HYPIR.enhancer.sd2 import SD2Enhancer


error_image = Image.open(os.path.join("assets", "gradio_error_img.png"))

# In this example, we set the max pixels to 2048x2048 to avoid processing images with excessively
# high resolution. Set max_size to None to remove this constraint.

max_size = (2048, 2048)
# max_size = None
if max_size is not None:
    print(f"Max size set to {max_size}, max pixels: {max_size[0] * max_size[1]}")
to_tensor = transforms.ToTensor()

model = SD2Enhancer(
    base_model_path="stabilityai/stable-diffusion-2-1-base",
    weight_path="HYPIR_sd2.pth",
    lora_modules=[
        "to_k",
        "to_q",
        "to_v",
        "to_out.0",
        "conv",
        "conv1",
        "conv2",
        "conv_shortcut",
        "conv_out",
        "proj_in",
        "proj_out",
        "ff.net.2",
        "ff.net.0.proj",
    ],
    lora_rank=256,
    model_t=200,
    coeff_t=200,
    device="cuda",
)
model.init_models()


def process(
    image,
    prompt,
    upscale,
    seed,
    progress=gr.Progress(track_tqdm=True),
):
    if seed == -1:
        seed = random.randint(0, 2**32 - 1)
    set_seed(seed)
    image = image.convert("RGB")
    # Check image size
    if max_size is not None:
        out_w, out_h = tuple(int(x * upscale) for x in image.size)
        if out_w * out_h > max_size[0] * max_size[1]:
            return error_image, (
                "Failed: The requested resolution exceeds the maximum pixel limit. "
                f"Your requested resolution is ({out_h}, {out_w}). "
                f"The maximum allowed pixel count is {max_size[0]} x {max_size[1]} "
                f"= {max_size[0] * max_size[1]} :("
            )

    image_tensor = to_tensor(image).unsqueeze(0)
    try:
        pil_image = model.enhance(
            lq=image_tensor,
            prompt=prompt,
            upscale=upscale,
            return_type="pil",
        )[0]
    except Exception as e:
        return error_image, f"Failed: {e} :("

    return pil_image, f"Success! :)\nUsed prompt: {prompt}"


block = gr.Blocks().queue()
with block:
    with gr.Row():
        with gr.Column():
            image = gr.Image(type="pil")
            prompt = gr.Textbox(label=("Prompt"))
            upscale = gr.Slider(minimum=1, maximum=8, value=1, label="Upscale Factor", step=1)
            seed = gr.Number(label="Seed", value=-1)
            run = gr.Button(value="Run")
        with gr.Column():
            result = gr.Image(type="pil", format="png")
            status = gr.Textbox(label="status", interactive=False)
        run.click(
            fn=process,
            inputs=[image, prompt, upscale, seed],
            outputs=[result, status],
        )
block.launch()